In [12]:
import os
import glob
import numpy as np
import pandas as pd

# from metabolights_utils.provider.study_provider import MetabolightsStudyProvider
from metabolights_utils.provider.ftp_repository import MetabolightsFtpRepository

# Select MetaboLights Studies

| Study ID | Focus | Samples (n) | Features (p) | Classification |
|----------|-------|-------------|--------------|----------------|
| MTBLS12968 | Polycystic Ovarian Syndrome (PCOS) | 149 | 486 | PCOS vs. healthy, term vs. preterm |
| MTBLS1 | Type 2 Diabetes | 132 | 220 | Diabetes vs. healthy |
| MTBLS2 | Arabidopsis thaliana (plant) | 16 | 15272 | 2 Genotypes |
| MTBLS8 | Yeast | 126 | 383 | ??? |



### **MTBLS12968**: PCOS preterm plasma metabolome

https://www.ebi.ac.uk/metabolights/editor/MTBLS12968/descriptors

*Xinyue HanPI*

This study investigated mid-trimester plasma metabolomic profiles in pregnant women to identify biomarkers associated with Polycystic Ovary Syndrome (PCOS) and preterm birth. A total of 149 participants were categorized into four groups: 32 with PCOS and preterm delivery, 34 with PCOS and term delivery, 42 without PCOS with preterm delivery, and 41 without PCOS with term delivery. Plasma samples were analyzed using targeted metabolomics via UHPLC-QTRAP MS/MS with HILIC and C18 chromatography. Multivariate statistical analyses including OPLS-DA were employed to identify significantly altered metabolites. The findings aim to provide insights into metabolic mechanisms underlying PCOS and preterm birth, potentially contributing to improved risk prediction strategies.


### **MTBLS1**: A metabolomic study of urinary changes in type 2 diabetes in human compared to the control group

https://www.ebi.ac.uk/metabolights/editor/MTBLS1/descriptors

*Reza M Salek, Jules L Griffin*

Type 2 diabetes mellitus is the result of a combination of impaired insulin secretion with reduced insulin sensitivity of target tissues. There are an estimated 150 million affected individuals worldwide, of whom a large proportion remains undiagnosed because of a lack of specific symptoms early in this disorder and inadequate diagnostics. In this study, NMR-based metabolomic analysis in conjunction with uni- and multivariate statistics was applied to examine the urinary metabolic changes in Human type 2 diabetes mellitus patients compared to the control group. The human population were un medicated diabetic patients who have good daily dietary control over their blood glucose concentrations by following the guidelines on diet issued by the American Diabetes Association. Note: This is part of a larger study, please refer to the original paper below.


### **MTBLS2**: Comparative LC/MS-based profiling of silver nitrate-treated Arabidopsis thaliana leaves of wild-type and cyp79B2 cyp79B3 double knockout plants

https://www.ebi.ac.uk/metabolights/editor/MTBLS2/descriptors

*Christoph Böttcher,  Steffen R Neumann*

Indole-3-acetaldoxime (IAOx) represents an early intermediate of the biosynthesis of a variety of indolic secondary metabolites including the phytoanticipin indol-3-ylmethyl glucosinolate and the phytoalexin camalexin (3-thiazol-2′-yl-indole). Arabidopsis thaliana cyp79B2 cyp79B3 double knockout plants are completely impaired in the conversion of tryptophan to indole-3-acetaldoxime and do not accumulate IAOx-derived metabolites any longer. Consequently, comparative analysis of wild-type and cyp79B2 cyp79B3 plant lines has the potential to explore the complete range of IAOx-derived indolic secondary metabolites.


### **MTBLS8**: Growth control of the eukaryote cell: a systems biology study in yeast

https://www.ebi.ac.uk/metabolights/editor/MTBLS8/descriptors

*Juan I Castrillo,  Stephen G Oliver,  Warwick B Dunn,  Neil Swainston,  Juan I Castrillo,  Leo A Zeef,  David C Hoyle,  Nianshu Zhang,  Andrew Hayes,  David Cj Gardner,  Michael J Cornell,  June Petty,  Luke Hakes,  Leanne Wardleworth,  Bharat Rash,  Marie Brown,  Warwick B Dunn,  David Broadhurst,  Kerry O'Donoghue*

Metabolic control analysis is being exploited in a systems biology study of the eukaryotic cell. Using chemostat culture, we have measured the impact of changes in flux (growth rate) on the transcriptome, proteome, endometabolome and exometabolome of the yeast Saccharomyces cerevisiae. Each functional genomic level shows clear growth-rate-associated trends and discriminates between carbon-sufficient and carbon-limited conditions. Genes consistently and significantly upregulated with increasing growth rate are frequently essential and encode evolutionarily conserved proteins of known function that participate in many protein-protein interactions. In contrast, more unknown, and fewer essential, genes are downregulated with increasing growth rate; their protein products rarely interact with one another. A large proportion of yeast genes under positive growth-rate control share orthologs with other eukaryotes, including humans. Significantly, transcription of genes encoding components of the TOR complex (a major controller of eukaryotic cell growth) is not subject to growth-rate regulation. Moreover, integrative studies reveal the extent and importance of post-transcriptional control, patterns of control of metabolic fluxes at the level of enzyme synthesis, and the relevance of specific enzymatic reactions in the control of metabolic fluxes during cell growth.

In [13]:
name2id = {
    "diabetes": "MTBLS1",
    "pcos": "MTBLS12968",  #  Y["control_vs_patient"] = Y["Factor Value[Group]"].str.split("-", n=1).str[0]
    "arabidopsis": "MTBLS2",
    "yeast": "MTBLS8",  # 126 samples × 383 metabolites ...cell growth control, not sure what the classes are (6?)
    # https://www.ebi.ac.uk/metabolights/editor/MTBLS8/descriptors
}

study_id = name2id["pcos"]

In [14]:
# 1. Load the study using FTP repository
repo = MetabolightsFtpRepository()
study, messages = repo.load_study_model(study_id)

# The study files are downloaded to a local directory
study_path = os.path.join(repo.local_storage_root_path, study_id)
print(f"Study downloaded to: {study_path}")

# 2. Get the Sample Metadata (Response Vector)
if study.samples:
    samples_file = list(study.samples.keys())[0]
    sample_df = pd.read_csv(os.path.join(study_path, samples_file), sep="\t")
    sample_df = sample_df.set_index("Sample Name")
    print(f"Loaded samples from {samples_file}: {sample_df.shape}")
else:
    raise ValueError("No samples file found in the study")

# 3. Search for all MAF (Metabolite Assignment File) TSV files in the directory
# Look for files containing "MAF" or metabolite-related keywords
maf_files = []
for pattern in ["*MAF*.tsv", "*metabolite*.tsv", "*_maf_*.tsv"]:
    maf_files.extend(glob.glob(os.path.join(study_path, pattern)))

# Remove duplicates
maf_files = list(set(maf_files))

if maf_files:
    print(f"Found {len(maf_files)} MAF file(s):")
    for f in maf_files:
        print(f"  - {os.path.basename(f)}")

    # Load and concatenate all MAF files
    all_features = []

    for maf_path in maf_files:
        maf_filename = os.path.basename(maf_path)

        # Load the MAF file
        data_matrix = pd.read_csv(maf_path, sep="\t")
        print(f"\nMAF loaded from {maf_filename}: {data_matrix.shape}")

        # Find sample columns by matching with sample names
        sample_cols = [col for col in data_matrix.columns if col in sample_df.index]

        if sample_cols:
            # Extract intensity data for this MAF
            X_maf = data_matrix[sample_cols]

            # Get feature names from the specified column if available
            feature_col = "metabolite_identification"
            if feature_col in data_matrix.columns:
                # Use the feature column as index
                X_maf.index = data_matrix[feature_col].values
                print(f"  Using '{feature_col}' as feature names")
            else:
                print(f"  Warning: '{feature_col}' not found, using numeric index")

            print(f"  Extracted {len(sample_cols)} samples × {X_maf.shape[0]} features")
            all_features.append(X_maf)
        else:
            print(f"  No sample columns found in {maf_filename}")

    if all_features:
        # Concatenate all MAF data matrices (along features/rows)
        X_raw = pd.concat(all_features, axis=0)

        # Transpose so samples are rows and features are columns
        X_raw = X_raw.T
        X_raw.index.name = "Sample Name"

        # Check if data is available
        non_nan_count = X_raw.notna().sum().sum()
        if non_nan_count == 0:
            print("\n⚠️  WARNING: No intensity values found in MAF files!")
            print(
                f"This study ({study_id}) may not have processed quantification data available."
            )
            print("Try a different study ID that has complete data.")
            X, Y = None, None
        else:
            X = X_raw.loc[sample_df.index.intersection(X_raw.index)]
            Y = sample_df.loc[X.index]

            print(f"\n✓ Successfully loaded data:")
            print(f"  X: {X.shape[0]} samples × {X.shape[1]} metabolites")
            print(f"  Y: {Y.shape[0]} samples × {Y.shape[1]} metadata attributes")
            print(f"  Non-missing values: {non_nan_count:,}")
    else:
        print(f"No valid data found in MAF files")
        X, Y = None, None
else:
    print("No MAF files found in the study directory")
    X, Y = None, None

Study downloaded to: C:\Users\Datamole\metabolights_data\studies\data\MTBLS12968
Loaded samples from s_MTBLS12968.txt: (149, 18)
Found 2 MAF file(s):
  - m_MTBLS12968_LC-MS_alternating_hilic_metabolite_profiling_v2_maf.tsv
  - m_MTBLS12968_LC-MS_alternating_reverse-phase_metabolite_profiling_v2_maf.tsv

MAF loaded from m_MTBLS12968_LC-MS_alternating_hilic_metabolite_profiling_v2_maf.tsv: (301, 170)
  Using 'metabolite_identification' as feature names
  Extracted 149 samples × 301 features

MAF loaded from m_MTBLS12968_LC-MS_alternating_reverse-phase_metabolite_profiling_v2_maf.tsv: (185, 170)
  Using 'metabolite_identification' as feature names
  Extracted 149 samples × 185 features

✓ Successfully loaded data:
  X: 149 samples × 486 metabolites
  Y: 149 samples × 18 metadata attributes
  Non-missing values: 49,141


In [15]:
# Display the loaded data
if X is not None and Y is not None:
    print("DATA SUMMARY")
    print("=" * 60)
    print(f"Samples: {X.shape[0]}")
    print(f"Metabolites: {X.shape[1]}")
    print(
        f"Missing values: {X.isna().sum().sum():,} ({100 * X.isna().sum().sum() / (X.shape[0] * X.shape[1]):.1f}%)"
    )

    # Show experimental factors
    factor_cols = [col for col in Y.columns if "Factor" in col]
    if factor_cols:
        print(f"\nExperimental Factors:")
        for col in factor_cols:
            print(f"  {col}:")
            print(f"    {Y[col].value_counts().to_dict()}")

    # Show characteristics
    char_cols = [col for col in Y.columns if "Characteristic" in col]
    if char_cols:
        print(f"\nCharacteristics:")
        for col in char_cols:
            print(f"  {col}:")
            print(f"    {Y[col].value_counts().to_dict()}")

    # Show other columns in metadata whose names don't contain "Factor" or "Characteristic"
    # but may still be relevant
    # and contain more than just unique values or only one value across all samples
    other_cols = [col for col in Y.columns if col not in factor_cols + char_cols]
    if other_cols:
        print(f"\nOther Interesting Metadata Columns:")
        for col in other_cols:
            if (Y[col].nunique() > 1) and (Y[col].nunique() < len(Y)):
                print(f"  {col}:")
                print(f"    {Y[col].value_counts().to_dict()}")

else:
    print("No data to display - please try a different study ID")

DATA SUMMARY
Samples: 149
Metabolites: 486
Missing values: 23,273 (32.1%)

Experimental Factors:
  Factor Value[Group]:
    {'HP-P1': 1, 'HP-P2': 1, 'HP-P3': 1, 'HP-P4': 1, 'HP-P5': 1, 'HP-P6': 1, 'HP-P7': 1, 'HP-P8': 1, 'HP-P9': 1, 'HP-P10': 1, 'HP-P11': 1, 'HP-P12': 1, 'HP-P13': 1, 'HP-P14': 1, 'HP-P15': 1, 'HP-P16': 1, 'HP-P18': 1, 'HP-P19': 1, 'HP-P20': 1, 'HP-P21': 1, 'HP-P22': 1, 'HP-P23': 1, 'HP-P24': 1, 'HP-P25': 1, 'HP-P26': 1, 'HP-P27': 1, 'HP-P28': 1, 'HP-P29': 1, 'HP-P30': 1, 'HP-P31': 1, 'HP-P32': 1, 'HP-P33': 1, 'HP-P34': 1, 'HP-P35': 1, 'HP-P36': 1, 'HP-P37': 1, 'HP-P38': 1, 'HP-P39': 1, 'HP-P40': 1, 'HP-P41': 1, 'HP-P42': 1, 'HP-P43': 1, 'HP-T1': 1, 'HP-T2': 1, 'HP-T3': 1, 'HP-T4': 1, 'HP-T5': 1, 'HP-T6': 1, 'HP-T7': 1, 'HP-T8': 1, 'HP-T9': 1, 'HP-T10': 1, 'HP-T11': 1, 'HP-T12': 1, 'HP-T13': 1, 'HP-T14': 1, 'HP-T15': 1, 'HP-T16': 1, 'HP-T17': 1, 'HP-T18': 1, 'HP-T19': 1, 'HP-T20': 1, 'HP-T21': 1, 'HP-T22': 1, 'HP-T23': 1, 'HP-T24': 1, 'HP-T25': 1, 'HP-T26': 1, 'HP-T27':

In [16]:
# Rename all unidentified columns in X to "unidentified_1", "unidentified_2", etc.
# Check for NaN, None, empty strings, or "Unnamed:" prefixes
unidentified_indices = []
for i, col in enumerate(X.columns):
    if (
        pd.isna(col)
        or col is None
        or col == ""
        or (isinstance(col, str) and col.startswith("Unnamed:"))
    ):
        unidentified_indices.append(i)

print(f"Found {len(unidentified_indices)} unidentified columns.")

# Rename them by position to handle duplicate column names
if unidentified_indices:
    new_columns = X.columns.tolist()
    for counter, idx in enumerate(unidentified_indices, 1):
        old_name = new_columns[idx]
        new_name = f"unidentified_{counter}"
        new_columns[idx] = new_name

    if len(unidentified_indices) > 10:
        print(f"  ... and {len(unidentified_indices) - 10} more")

    X.columns = new_columns
    print(f"✓ Successfully renamed {len(unidentified_indices)} columns")
else:
    print("No unidentified columns found")

Found 0 unidentified columns.
No unidentified columns found


In [17]:
X.sample(10)

,4-Pyridoxic acid,Tryptophanamide,Nicotinamide,Aniline-2-sulfonate,"3,4-Dihydroxyphenylglycol",2'-O-methyladenosine,Thymidine,Uracil,Hordenine,Deoxyuridine,...,Cholesterol sulfate,Stearic acid,11Z-Eicosenoic acid,"13Z,16Z-Docosadienoic acid",Arachidic acid,Erucic acid,Docosanoic acid,Tricosanoic acid,Glycerol-myristate,Didecanoylphosphatidylcholine
Sample Name,,,,,,,,,,,,,,,,,,,,,
PCOS-T18,9.346760,12.219418,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,203.780849,9640.631309,7709.211206,9.525068,NaN,123.133657,24047.63611,NaN,15.383558,0.011795
HP-T16,4.095071,13.889366,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1094.790655,16552.809180,36037.972430,NaN,NaN,597.707661,68888.73211,NaN,507.706219,NaN
HP-T19,2.833259,12.493324,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,155.541148,9909.041409,11299.952300,15.044138,15330.89204,287.435876,56159.91646,180804.9385,1.969902,NaN
HP-P26,10.841021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,166.500985,11026.811250,10852.993310,NaN,NaN,NaN,24403.03591,NaN,75.740609,0.048722
PCOS-T21,4.301491,12.911536,26.675032,NaN,NaN,NaN,35.894023,NaN,NaN,NaN,...,274.374331,10838.496020,7572.238103,12.558564,NaN,NaN,21616.54591,103099.4259,19.563777,0.017543
HP-T22,NaN,12.988573,32.966891,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,263.527028,19711.305330,49487.958500,NaN,NaN,2389.079060,52446.71567,NaN,323.733864,NaN
PCOS-T19,7.372042,12.511319,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,115.399641,14527.111750,22204.596490,23.089053,14225.26452,433.312049,24503.95881,NaN,62.464264,NaN
HP-T18,4.622444,11.544200,41.024109,NaN,NaN,NaN,23.850498,20.327888,NaN,NaN,...,1059.150707,14247.462150,14587.208090,25.347081,NaN,NaN,38742.00983,152891.0550,30.038748,NaN
HP-P19,9.480895,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,263.239154,63036.689080,144901.701900,767.829932,NaN,4593.178111,61764.82981,NaN,159.079588,NaN


In [18]:
Y.sample(10)

,Source Name,Characteristics[Organism],Term Source REF,Term Accession Number,Characteristics[Organism part],Term Source REF.1,Term Accession Number.1,Characteristics[Variant],Term Source REF.2,Term Accession Number.2,Characteristics[Sample type],Term Source REF.3,Term Accession Number.3,Protocol REF,Factor Value[Group],Unit,Term Source REF.4,Term Accession Number.4
Sample Name,,,,,,,,,,,,,,,,,,
HP-P23,HP-P23,Homo sapiens,NCBITaxon,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,blood plasma,MTBLS,http://www.ebi.ac.uk/metabolights/ontology/pla...,Sample collection,HP-P23,Group,Group,http://xmlns.com/foaf/0.1/Group
PCOS-P26,PCOS-P26,Homo sapiens,NCBITaxon,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,blood plasma,MTBLS,http://www.ebi.ac.uk/metabolights/ontology/pla...,Sample collection,PCOS-P26,Group,Group,http://xmlns.com/foaf/0.1/Group
PCOS-T23,PCOS-T23,Homo sapiens,NCBITaxon,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,blood plasma,MTBLS,http://www.ebi.ac.uk/metabolights/ontology/pla...,Sample collection,PCOS-T23,Group,Group,http://xmlns.com/foaf/0.1/Group
PCOS-T10,PCOS-T10,Homo sapiens,NCBITaxon,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,blood plasma,MTBLS,http://www.ebi.ac.uk/metabolights/ontology/pla...,Sample collection,PCOS-T10,Group,Group,http://xmlns.com/foaf/0.1/Group
HP-P39,HP-P39,Homo sapiens,NCBITaxon,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,blood plasma,MTBLS,http://www.ebi.ac.uk/metabolights/ontology/pla...,Sample collection,HP-P39,Group,Group,http://xmlns.com/foaf/0.1/Group
PCOS-T24,PCOS-T24,Homo sapiens,NCBITaxon,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,blood plasma,MTBLS,http://www.ebi.ac.uk/metabolights/ontology/pla...,Sample collection,PCOS-T24,Group,Group,http://xmlns.com/foaf/0.1/Group
PCOS-T9,PCOS-T9,Homo sapiens,NCBITaxon,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,blood plasma,MTBLS,http://www.ebi.ac.uk/metabolights/ontology/pla...,Sample collection,PCOS-T9,Group,Group,http://xmlns.com/foaf/0.1/Group
PCOS-P32,PCOS-P32,Homo sapiens,NCBITaxon,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,blood plasma,MTBLS,http://www.ebi.ac.uk/metabolights/ontology/pla...,Sample collection,PCOS-P32,Group,Group,http://xmlns.com/foaf/0.1/Group
PCOS-P18,PCOS-P18,Homo sapiens,NCBITaxon,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,blood plasma,MTBLS,http://www.ebi.ac.uk/metabolights/ontology/pla...,Sample collection,PCOS-P18,Group,Group,http://xmlns.com/foaf/0.1/Group


In [19]:
sample_name_prefix = Y.index.str.split("_", n=1).str[0]
sample_name_prefix.value_counts()

Sample Name
HP-P1       1
HP-P2       1
HP-P3       1
HP-P4       1
HP-P5       1
           ..
PCOS-T30    1
PCOS-T31    1
PCOS-T32    1
PCOS-T33    1
PCOS-T34    1
Name: count, Length: 149, dtype: int64

## Save as CSV

In [20]:
df = X.copy()

# Add experimental factors as features - includes potential response variable(s)
# Add characteristics as features if they have more than 1 unique value
# Add other metadata columns as features
#   if they have more than 1 unique value and less than total number of samples
for col in char_cols + factor_cols + other_cols:
    n_unique = Y[col].nunique()
    if (n_unique > 1) and (n_unique < len(Y)):
        print(
            f"Adding metadata column '{col}' as a feature (unique values: {n_unique})"
        )
        df[f"metadata__{col}"] = Y[col]


In [21]:
# PCOS study has "Factor Value[Group]" that contains both stratification and response information
# But it contains unique patient numbers
# 4 groups:
#   "HP-T" ... control, term delivery,
#   "HP-P" ... control, preterm delivery,
#   "PCOS-T" ... PCOS, term delivery,
#   "PCOS-P" ... PCOS, preterm delivery
# create columns:
#   metadata__PCOS ... contains values: PCOS vs control
#   metadata__PRETERM ... contains values: preterm vs term
if study_id == "MTBLS12968":
    col = "Factor Value[Group]"
    df["metadata__PCOS"] = (
        Y[col].str.contains("PCOS").apply(lambda x: "pcos" if x else "control")
    )
    df["metadata__PRETERM"] = (
        Y[col].str.contains("-P").apply(lambda x: "preterm" if x else "term")
    )


In [22]:
df.to_csv(f"../data/metabolights/metabolights_{study_id}.csv", index=True)